# Current C00 pilot audit

## tl;dr

This companion reproduces the integrity and metric checks used in the review of the only current result, C00:

- state continuity and duplicate/missing `agent × date × subturn` records;
- treatment delivery, belief coverage, and forced buy/sell mechanics;
- validation metrics alongside the repository's simple baselines;
- persona-population versus active-cohort coverage.

The remaining five information-environment conditions are a proposed future factorial design, not observed experimental results in this notebook.

## Context & Methods

### Key Assumptions

- The C00 root `outputs/logs/<run_id>/agent_turns.jsonl` file is the authoritative execution log.
- Its matching `validation/outputs/<run_id>/summary_metrics.json` file is the authoritative source for reported direction metrics.
- A valid continuous 63-day run has 30 agents × 63 dates × AM/PM = 3,780 unique agent-turn keys and a global turn sequence spanning 1–126.
- Fake `read` and `selected` indicators are post-treatment mechanisms. They are described but never conditioned on as causal controls.

In [1]:
from pathlib import Path
import subprocess
import sys
import pandas as pd

project_root = Path.cwd()
if not (project_root / 'analysis' / 'current_experiment_review').exists():
    raise RuntimeError('Run this notebook from the repository root.')

subprocess.run(
    [sys.executable, 'analysis/current_experiment_review/analyze_current_runs.py'],
    cwd=project_root,
    check=True,
)
artifact_dir = project_root / 'analysis/current_experiment_review/outputs'

Wrote audit artifacts to /Users/kwon_junyoung/Documents/AICP/2026_AICP_ver2.0/analysis/current_experiment_review/outputs


## Data

In [2]:
audit = pd.read_csv(artifact_dir / 'run_integrity_audit.csv')
benchmarks = pd.read_csv(artifact_dir / 'validation_benchmarks.csv')
cohort = pd.read_csv(artifact_dir / 'persona_population_vs_active_cohort.csv')
persona_descriptives = pd.read_csv(artifact_dir / 'c00_persona_behavior_descriptives.csv')
log_inventory = pd.read_csv(artifact_dir / 'c00_log_inventory.csv')
daily_decisions = pd.read_csv(artifact_dir / 'c00_daily_decision_summary.csv')
chunk_boundaries = pd.read_csv(artifact_dir / 'c00_chunk_boundary_summary.csv')
validation_by_chunk = pd.read_csv(artifact_dir / 'c00_validation_by_chunk.csv')

audit[['condition', 'label', 'raw_rows', 'unique_agent_date_subturn_rows',
       'duplicate_key_rows', 'missing_key_rows', 'turn_max',
       'restart_date_count', 'fake_exposure_slots', 'blank_belief_rate',
       'fallback_decision_rows', 'fallback_sell_orders']]

,condition,label,raw_rows,unique_agent_date_subturn_rows,duplicate_key_rows,missing_key_rows,turn_max,restart_date_count,fake_exposure_slots,blank_belief_rate,fallback_decision_rows,fallback_sell_orders
0,C00,community off · fake off,3780,3780,0,0,10,13,0,0.2024,50,48


## Results

In [3]:
metric_columns = ['condition', 'direction_match_rate', 'balanced_accuracy',
                  'buy_recall', 'sell_recall', 'daily_pearson',
                  'validation_overlap_days']
audit[metric_columns].sort_values('condition')

,condition,direction_match_rate,balanced_accuracy,buy_recall,sell_recall,daily_pearson,validation_overlap_days
0,C00,0.6207,0.5784,0.8235,0.3333,0.120739,58


### C00 log map

In [4]:
log_inventory.groupby(['surface', 'scope', 'category', 'grain', 'interpretation_use'], as_index=False).agg(
    files=('path', 'count'),
    total_bytes=('bytes', 'sum')
).sort_values(['surface', 'scope', 'category'])

,surface,scope,category,grain,interpretation_use,files,total_bytes
0,run,chunk copy,community trace,post/read/reaction,should be empty or irrelevant under C00 commun...,65,5785
1,run,chunk copy,completion marker,run,whether the recorded run completed,13,3952
2,run,chunk copy,daily market summary,date × subturn,"aggregate order/fill counts, price, and volume",26,3359313
3,run,chunk copy,decision trace,agent × AM/PM turn,"news/context, belief, rationale, action, quantity",26,76927189
4,run,chunk copy,fills,executed fill,"prices, quantities, and execution",13,321281
5,run,chunk copy,orders,submitted order,requested buy/sell orders before fills,13,1410993
6,run,chunk copy,portfolio state,agent × post-fill turn,"cash, holdings, asset value, and PnL",13,2376511
7,run,chunk copy,run specification,run,"period, condition flags, concurrency, and inpu...",13,24387
8,run,root,checkpoint,chunk,resume state; not a behavioral outcome,1,322
9,run,root,community trace,post/read/reaction,should be empty or irrelevant under C00 commun...,5,445


In [5]:
# Root-level sources are the joined reading path.  Chunk copies are useful for detecting restart artifacts.
log_inventory.loc[log_inventory['scope'].eq('root'),
                  ['path', 'category', 'grain', 'interpretation_use', 'line_count']]

,path,category,grain,interpretation_use,line_count
0,outputs/logs/simulation_20260715_30agents_comm...,decision trace,agent × AM/PM turn,"news/context, belief, rationale, action, quantity",3781.0
1,outputs/logs/simulation_20260715_30agents_comm...,decision trace,agent × AM/PM turn,"news/context, belief, rationale, action, quantity",3780.0
2,outputs/logs/simulation_20260715_30agents_comm...,checkpoint,chunk,resume state; not a behavioral outcome,19.0
3,outputs/logs/simulation_20260715_30agents_comm...,runtime/checkpoint state,internal,recovery artifact; inspect only for integrity ...,NaN
4,outputs/logs/simulation_20260715_30agents_comm...,runtime/checkpoint state,internal,recovery artifact; inspect only for integrity ...,NaN
5,outputs/logs/simulation_20260715_30agents_comm...,runtime/checkpoint state,internal,recovery artifact; inspect only for integrity ...,NaN
6,outputs/logs/simulation_20260715_30agents_comm...,runtime/checkpoint state,internal,recovery artifact; inspect only for integrity ...,NaN
7,outputs/logs/simulation_20260715_30agents_comm...,runtime/checkpoint state,internal,recovery artifact; inspect only for integrity ...,NaN
8,outputs/logs/simulation_20260715_30agents_comm...,runtime/checkpoint state,internal,recovery artifact; inspect only for integrity ...,NaN
9,outputs/logs/simulation_20260715_30agents_comm...,runtime/checkpoint state,internal,recovery artifact; inspect only for integrity ...,NaN


### State continuity and decision trace

In [6]:
chunk_boundaries

,chunk,agent_turn_rows,date_min,date_max,turn_min,turn_max,first_am_rows,first_am_initial_portfolio_rows,first_am_buy_orders
0,chunk_001_2026-02-27_2026-03-06,300,2026-02-27,2026-03-06,1,10,30,30,30
1,chunk_002_2026-03-09_2026-03-13,300,2026-03-09,2026-03-13,1,10,30,30,30
2,chunk_003_2026-03-16_2026-03-20,300,2026-03-16,2026-03-20,1,10,30,30,30
3,chunk_004_2026-03-23_2026-03-27,300,2026-03-23,2026-03-27,1,10,30,30,30
4,chunk_005_2026-03-30_2026-04-03,300,2026-03-30,2026-04-03,1,10,30,30,30
5,chunk_006_2026-04-06_2026-04-10,300,2026-04-06,2026-04-10,1,10,30,30,30
6,chunk_007_2026-04-13_2026-04-17,300,2026-04-13,2026-04-17,1,10,30,30,30
7,chunk_008_2026-04-20_2026-04-24,300,2026-04-20,2026-04-24,1,10,30,30,30
8,chunk_009_2026-04-27_2026-05-04,300,2026-04-27,2026-05-04,1,10,30,30,30
9,chunk_010_2026-05-06_2026-05-12,300,2026-05-06,2026-05-12,1,10,30,30,30


In [7]:
daily_decisions[['date', 'turn', 'subturn', 'agent_rows', 'buy_orders', 'sell_orders',
                 'net_submitted_quantity', 'initial_portfolio_agents', 'blank_belief_rows',
                 'positive_news_sentiment', 'negative_news_sentiment', 'mixed_news_sentiment']]

,date,turn,subturn,agent_rows,buy_orders,sell_orders,net_submitted_quantity,initial_portfolio_agents,blank_belief_rows,positive_news_sentiment,negative_news_sentiment,mixed_news_sentiment
0,2026-02-27,1,am,30,30,0,6536,30,4,10,0,20
1,2026-02-27,2,pm,30,23,7,1215,0,5,6,0,24
2,2026-03-03,3,am,30,30,0,2802,0,6,28,0,1
3,2026-03-03,4,pm,30,23,7,662,0,6,20,0,10
4,2026-03-04,5,am,30,26,4,662,0,9,0,2,27
...,...,...,...,...,...,...,...,...,...,...,...,...
121,2026-05-28,2,pm,30,20,10,402,0,3,0,2,25
122,2026-05-29,3,am,30,20,10,731,0,8,10,0,18
123,2026-05-29,4,pm,30,10,20,-427,0,3,1,0,28
124,2026-06-01,5,am,30,15,15,210,0,8,3,0,24


In [8]:
benchmarks.loc[
    benchmarks['benchmark'].isin(['always_buy', 'previous_day_market_return_direction']),
    ['condition', 'benchmark', 'direction_match_rate', 'balanced_accuracy',
     'buy_recall', 'sell_recall']
].sort_values(['condition', 'benchmark'])

,condition,benchmark,direction_match_rate,balanced_accuracy,buy_recall,sell_recall
0,C00,always_buy,0.586207,0.500000,1.000000,0.000000
5,C00,previous_day_market_return_direction,0.568966,0.564951,0.588235,0.541667


In [9]:
# These are not independent replications.  They expose the variability
# concealed by a single 58-day aggregate while each chunk restarts state.
validation_by_chunk

,condition,chunk,validation_days,date_min,date_max,direction_match_rate,predicted_buy_days,predicted_sell_days,actual_buy_days,actual_sell_days,buy_recall,sell_recall,balanced_accuracy
0,C00,chunk_002_2026-03-09_2026-03-13,5,2026-03-09,2026-03-13,0.8000,5,0,4,1,1.0000,0.0000,0.5000
1,C00,chunk_003_2026-03-16_2026-03-20,5,2026-03-16,2026-03-20,0.6000,4,1,2,3,1.0000,0.3333,0.6667
2,C00,chunk_004_2026-03-23_2026-03-27,5,2026-03-23,2026-03-27,0.8000,4,1,5,0,0.8000,NaN,0.8000
3,C00,chunk_005_2026-03-30_2026-04-03,5,2026-03-30,2026-04-03,0.8000,2,3,3,2,0.6667,1.0000,0.8333
4,C00,chunk_006_2026-04-06_2026-04-10,5,2026-04-06,2026-04-10,0.6000,4,1,2,3,1.0000,0.3333,0.6667
5,C00,chunk_007_2026-04-13_2026-04-17,5,2026-04-13,2026-04-17,0.4000,3,2,2,3,0.5000,0.3333,0.4167
6,C00,chunk_008_2026-04-20_2026-04-24,5,2026-04-20,2026-04-24,0.4000,4,1,3,2,0.6667,0.0000,0.3333
7,C00,chunk_009_2026-04-27_2026-05-04,5,2026-04-27,2026-05-04,0.6000,4,1,2,3,1.0000,0.3333,0.6667
8,C00,chunk_010_2026-05-06_2026-05-12,5,2026-05-06,2026-05-12,0.6000,4,1,4,1,0.7500,0.0000,0.3750
9,C00,chunk_011_2026-05-13_2026-05-19,5,2026-05-13,2026-05-19,0.6000,4,1,4,1,0.7500,0.0000,0.3750


In [10]:
cohort.loc[cohort['field'].isin(['age_group', 'ini_cash', 'strategy', 'news_depth'])]

,field,value,population_count,population_share,active_cohort_count,active_cohort_share
2,age_group,20대,9,0.09,9,0.3000
3,age_group,30대,18,0.18,18,0.6000
4,age_group,40대,23,0.23,3,0.1000
5,age_group,50대,26,0.26,0,0.0000
6,age_group,60대,17,0.17,0,0.0000
7,age_group,70대,6,0.06,0,0.0000
8,age_group,80대 이상,1,0.01,0,0.0000
9,ini_cash,100000000,90,0.90,30,1.0000
10,ini_cash,1000000000,10,0.10,0,0.0000
11,strategy,technical,58,0.58,16,0.5333


In [11]:
# Descriptive only: this is not a persona-effect estimate because C00 has
# 30 homogeneous-capital agents and several sparse persona cells.
persona_descriptives.loc[
    persona_descriptives['persona_field'].isin(['age_group', 'strategy', 'news_depth']),
    ['persona_field', 'persona_value', 'agent_count', 'buy_share', 'sell_share',
     'one_share_rate', 'blank_belief_rate']
]

,persona_field,persona_value,agent_count,buy_share,sell_share,one_share_rate,blank_belief_rate
0,age_group,20대,9,0.6173,0.3827,0.0265,0.2063
1,age_group,30대,18,0.6574,0.3426,0.0287,0.2024
2,age_group,40대,3,0.6984,0.3016,0.0794,0.1905
6,news_depth,0,10,0.6571,0.3429,0.0468,0.2040
7,news_depth,1,16,0.6597,0.3403,0.0263,0.1984
8,news_depth,2,4,0.5893,0.4107,0.0258,0.2143
9,strategy,technical,16,0.6726,0.3274,0.0283,0.2004
10,strategy,value,14,0.6230,0.3770,0.0385,0.2046


## Takeaways

Interpret any numerical difference only after the execution-integrity table is clean.  In particular, a repeated local turn range or a reset to the initial portfolio means a multi-day trajectory, memory effect, return path, and aggregate-flow comparison cannot be treated as the intended continuous experiment.  Missing belief text and incomplete fake delivery also bound which embedding or rubric analyses are usable.